In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score
import seaborn as sbn
import random

In [ ]:
random.seed(42)

In [ ]:
def train_and_eval(data, labels):
    x_train, x_val, y_train, y_val = train_test_split(data,labels, random_state=42)
    clf = SVC()
    clf.fit(x_train, y_train)
    y_pred = clf.predict(x_val)
    print("\tPrecision: %1.3f" % precision_score(y_val, y_pred))
    print("\tRecall: %1.3f" % recall_score(y_val, y_pred))
    return clf

In [ ]:
def train_and_eval_rf(data, labels):
    x_train, x_val, y_train, y_val = train_test_split(data,labels, random_state=42)
    clf = RandomForestClassifier()
    clf.fit(x_train, y_train)
    y_pred = clf.predict(x_val)
    print("\tPrecision: %1.3f" % precision_score(y_val, y_pred))
    print("\tRecall: %1.3f" % recall_score(y_val, y_pred))
    return clf

In [ ]:
df = pd.read_csv("bank-additional-full.csv", sep=";")
g = df.groupby('y')
# On ré-equilibre le jeu de donnée pour que l'on ait pas de soucis dans le reste du tp du au desequilibre
df = g.apply(lambda x: x.sample(g.size().min()).reset_index(drop=True)).reset_index(drop=False)
df.head(10) # affiche les 10 premières lignes

In [ ]:
df['y'].value_counts(normalize=True)

In [ ]:
df.describe()

### Séparons et encodons nos labels

In [ ]:
y = df['y'].values
y

In [ ]:
y.shape

Quel type choisir ?

In [ ]:
# TODO

N'oublions pas de supprimer la colonne label des données envoyées au modele

In [ ]:
df = df.drop(columns=['y'])
df.columns

### Encodons quelques colonnes

#### Affichons les colonnes `job` et `education`

In [ ]:
plot = sbn.displot(df['job'])
plot.set_xticklabels(rotation=45)

In [ ]:
plot = sbn.displot(df['education'])
plot.set_xticklabels(rotation=45)

#### Encodons ces deux colonnes et testons de prédire `y` avec :

In [ ]:
# TODO

In [ ]:
clf = train_and_eval(jobs_ord, y)

#### Et avec un `OneHotEncoding` ?

In [ ]:
# TODO

#### Regardons le premier élément

In [ ]:
jobs[0]

In [ ]:
print(jobs_onehot[0])

### Entrainons sur ces valeurs

In [ ]:
clf = train_and_eval(jobs_onehot, y)

In [ ]:
rf = train_and_eval_rf(jobs_onehot, y)

In [ ]:
rf = train_and_eval_rf(jobs_ord, y)

On obtient de meilleurs résulats avec l'`OrdinalEncoding`. Et avec un `RandomForest` ?

#### Rajoutons maintenant la colonne `age`

In [ ]:
plot = sbn.displot(df["age"], bins=range(max(df.age)))
plot.set_xticklabels(rotation=45)

In [ ]:
n_sample = df.shape[0]
train_data = np.empty((n_sample, jobs_ord.shape[1]+1)) # crée un nouveau ndarray ayant n_sample lignes et 3 colonnes (jobs_ord a 2 colonnes))
train_data[:,:-1] = jobs_ord # les deux premières colonnes correspondent à jobs_ord
train_data[:,-1] = df["age"] # la dernière est la colonne age

In [ ]:
train_data

In [ ]:
clf = train_and_eval(train_data, y)

On observe une chute du `recall`
... mais une colonne a peut-<être biaisé le modele comme elle a une plage de valeur différente...

In [ ]:
train_data[0]

### Normalisation des données

 - On ne va pas utiliser un `OneHotEncoder` comme l'âge est une grandeur ordonnée
 - Attention : [`Normalizer`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.Normalizer.html) ne fait pas ce que vous voulez, il normalise les échantillons (les lignes) au lieu de normaliser les _features_

In [ ]:
# TODO

In [ ]:
train_data_norm[0]

In [ ]:
clf = train_and_eval(train_data_norm, y)

$\Rightarrow$ l'intérêt de la normalisation est évident ici

### Testons la standardisation, comme l'âge ressemble à une distribution normale

In [ ]:
# TODO

In [ ]:
sbn.displot(train_data_std[:,-1],bins=np.arange(-3,5,0.1))
plt.title("Standardization")
sbn.displot(train_data_norm[:,-1],bins=np.arange(0,1,0.02))
_ = plt.title("Normalization")

In [ ]:
clf = train_and_eval(train_data_std, y)

- la standardisation n'a rien apporté
- on remarque que la distribution n'est pas bien répartie autour de 0, testons des méthodes plus avancées de normalisation

In [ ]:
# TODO

In [ ]:
# TODO

In [ ]:
# TODO

### Et si on avait eu du texte ?

In [ ]:
mon_texte = "une belle chaine chaine de caractere comme notre jeu de donnée n'en avait pas"

In [ ]:
# TODO

In [ ]:
mon_texte_encode

- Et pour savoir quelle feature encode quel mot ?

In [ ]:
count_vec.get_feature_names_out()

$\Rightarrow$ l'ordre n'est donc pas respecté

In [ ]:
[(w, c) for w, c in zip(np.asarray(mon_texte_encode)[0], count_vec.get_feature_names_out())]